### Imports

In [90]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import datasets
import numpy
import os
import torch.nn as nn
from transformers import RobertaTokenizer, RobertaForSequenceClassification,  Trainer, TrainingArguments, DataCollatorWithPadding

### Load and tokenize data

In [91]:
model_name = 'distilroberta-base'
# model_name = 'roberta-base'
# model_name = 'roberta-small'

In [92]:
tokenizer = RobertaTokenizer.from_pretrained(model_name)

In [93]:
values = [ "Self-direction: thought", "Self-direction: action", "Stimulation",  "Hedonism", "Achievement", "Power: dominance", "Power: resources", "Face", "Security: personal", "Security: societal", "Tradition", "Conformity: rules", "Conformity: interpersonal", "Humility", "Benevolence: caring", "Benevolence: dependability", "Universalism: concern", "Universalism: nature", "Universalism: tolerance" ]
labels = sum([[value + " attained", value + " constrained"] for value in values], [])

In [94]:
num_labels = len(labels)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)  

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [95]:
# Fixing weights and adding a final classification layer to train

class CustomRobertaForSequenceClassification(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()
        self.roberta = base_model.roberta  # Use the RoBERTa backbone
        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(base_model.config.hidden_size, num_labels),
        )

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_output)

        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(logits, labels)

        return (loss, logits) if loss is not None else logits


for param in model.parameters():
    param.requires_grad = False
    
model = CustomRobertaForSequenceClassification(model, num_labels=len(labels))

In [96]:
def load_dataset(directory, tokenizer, load_labels=True):
    sentences_file_path = os.path.join(directory, "sentences.tsv")
    labels_file_path = os.path.join(directory, "labels.tsv")
    
    data_frame = pd.read_csv(sentences_file_path, encoding="utf-8", sep="\t", header=0)
    encoded_sentences = tokenizer(data_frame["Text"].to_list(), truncation=True)

    if load_labels and os.path.isfile(labels_file_path):
        labels_frame = pd.read_csv(labels_file_path, encoding="utf-8", sep="\t", header=0)
        labels_frame = pd.merge(data_frame, labels_frame, on=["Text-ID", "Sentence-ID"])
        labels_matrix = numpy.zeros((labels_frame.shape[0], len(labels)))
        for idx, label in enumerate(labels):
            if label in labels_frame.columns:
                labels_matrix[:, idx] = (labels_frame[label] >= 0.5).astype(int)
        encoded_sentences["labels"] = labels_matrix.tolist()

    encoded_sentences = datasets.Dataset.from_dict(encoded_sentences)
    
    return encoded_sentences, data_frame["Text-ID"].to_list(), data_frame["Sentence-ID"].to_list()

In [97]:
directory_test="datasets/valueeval24/test-english"
directory_train="datasets/valueeval24/training-english"
directory_validation="datasets/valueeval24/validation-english"

encoded_sentences_test, text_ids_test, sentence_ids_test = load_dataset(directory_test, tokenizer)
encoded_sentences_train, text_ids_train, sentence_ids_train = load_dataset(directory_train, tokenizer)
encoded_sentences_validation, text_ids_validation, sentence_ids_validation = load_dataset(directory_validation, tokenizer)

### Training parameters

In [98]:
def compute_metrics(pred):
    labels = pred.label_ids
    # Prendre les probabilités prédictives et les convertir en labels binaires avec un seuil de 0.5
    preds = (pred.predictions >= 0.5).astype(int)
    
    # Calcul des métriques pour un problème multi-label
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='samples',zero_division=0)
    accuracy = accuracy_score(labels, preds)
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [99]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [100]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,             
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    fp16=True
)

/Users/edabier/miniconda3/envs/roberta_env/lib/python3.11/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [101]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_sentences_train,
    eval_dataset=encoded_sentences_test, 
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [102]:
model

CustomRobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
     

In [ ]:
trainer.train()

  0%|          | 0/33570 [00:00<?, ?it/s]

In [ ]:
test_metrics = trainer.evaluate(encoded_sentences_validation)
print("Évaluation sur le jeu de test:", test_metrics)